In [14]:
import pandas as pd
from pathlib import Path
import sys

sys.path.append("../")

from src.data_utils import (
    snake_columns,
    convert_to_datetime,
    create_review_text,
    add_delivery_features
)

In [15]:
RAW_PATH = Path("../data/raw")
PROCESSED_PATH = Path("../data/processed")

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

In [16]:
def snake_columns(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )
    return df

In [17]:
reviews = pd.read_csv(RAW_PATH / "olist_order_reviews_dataset.csv")
orders = pd.read_csv(RAW_PATH / "olist_orders_dataset.csv")
customers = pd.read_csv(RAW_PATH / "olist_customers_dataset.csv")
payments = pd.read_csv(RAW_PATH / "olist_order_payments_dataset.csv")

In [18]:
reviews = snake_columns(reviews)
orders = snake_columns(orders)
customers = snake_columns(customers)
payments = snake_columns(payments)

In [19]:
print("REVIEWS")
print(reviews.columns.tolist())

print("\nORDERS")
print(orders.columns.tolist())

print("\nCUSTOMERS")
print(customers.columns.tolist())

print("\nPAYMENTS")
print(payments.columns.tolist())

REVIEWS
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

ORDERS
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

CUSTOMERS
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

PAYMENTS
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']


In [20]:
orders_date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in orders_date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")


reviews_date_cols = [
    "review_creation_date",
    "review_answer_timestamp"
]

for col in reviews_date_cols:
    reviews[col] = pd.to_datetime(reviews[col], errors="coerce")

In [21]:
reviews["review_comment_title"] = reviews["review_comment_title"].fillna("")
reviews["review_comment_message"] = reviews["review_comment_message"].fillna("")

reviews["review_text"] = (
    reviews["review_comment_title"] + " " + reviews["review_comment_message"]
).str.strip()

reviews["has_review_text"] = reviews["review_text"].str.len() > 0

In [22]:
orders["delivery_time_days"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.days

orders["estimated_delivery_days"] = (
    orders["order_estimated_delivery_date"] - orders["order_purchase_timestamp"]
).dt.days

orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]
).dt.days

orders["is_late"] = orders["delivery_delay_days"] > 0

In [23]:
print("Reviews shape:", reviews.shape)
print("Orders shape:", orders.shape)
print("Customers shape:", customers.shape)
print("Payments shape:", payments.shape)

print("\nReviews with text:")
print(reviews["has_review_text"].value_counts())

print("\nOrder status:")
print(orders["order_status"].value_counts())

Reviews shape: (99224, 9)
Orders shape: (99441, 12)
Customers shape: (99441, 5)
Payments shape: (103886, 5)

Reviews with text:
has_review_text
False    56537
True     42687
Name: count, dtype: int64

Order status:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [24]:
reviews.to_csv(PROCESSED_PATH / "reviews_clean.csv", index=False)
orders.to_csv(PROCESSED_PATH / "orders_clean.csv", index=False)
customers.to_csv(PROCESSED_PATH / "customers_clean.csv", index=False)
payments.to_csv(PROCESSED_PATH / "payments_clean.csv", index=False)

In [25]:
print("Saved files:")
print(PROCESSED_PATH / "reviews_clean.csv")
print(PROCESSED_PATH / "orders_clean.csv")
print(PROCESSED_PATH / "customers_clean.csv")
print(PROCESSED_PATH / "payments_clean.csv")

Saved files:
../data/processed/reviews_clean.csv
../data/processed/orders_clean.csv
../data/processed/customers_clean.csv
../data/processed/payments_clean.csv
